## 八、综合案例：设备故障风险识别

### 8.1 案例背景与数据

机房运维团队希望在每天巡检时，预测设备未来 7 天内是否会发生故障：

- 标签 `1`：未来 7 天内发生故障，是正类；
- 标签 `0`：未来 7 天内未发生故障，是负类；
- 漏掉故障设备的代价更高，因此主要关注 Recall；
- 业务要求验证集 `Recall >= 0.60`，满足要求后再选择 Precision 较高的阈值。

数据文件：[03_device_failures.csv](data/03_device_failures.csv)

| 字段                       | 含义           | 是否作为特征           |
| ------------------------ | ------------ | ---------------- |
| `device_id`              | 设备编号         | 否，仅用于定位样本        |
| `device_type`            | 设备类型         | 是                |
| `room_zone`              | 机房区域         | 是                |
| `temperature_c`          | 当前温度         | 是                |
| `vibration_mm_s`         | 当前振动值        | 是                |
| `operating_hours`        | 累计运行时长       | 是                |
| `error_count_7d`         | 最近 7 天错误次数   | 是                |
| `days_since_maintenance` | 距离上次维护的天数    | 是                |
| `downtime_hours_next_7d` | 未来 7 天实际停机时长 | 否，属于未来信息，会造成数据泄漏 |
| `failure_within_7d`      | 未来 7 天是否故障   | 否，这是预测标签         |

### 8.2 起始代码

In [17]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [18]:
# 读取 03 独立案例数据
data_path = Path('data/03_device_failures.csv')
devices = pd.read_csv(data_path)

In [19]:
print(devices.head())
print("数据形状：", devices.shape)
print("缺失值数量：")
print(devices.isna().sum())
print("重复行数量：", devices.duplicated().sum())
print("标签分布：")
print(devices["failure_within_7d"].value_counts().sort_index())

  device_id device_type room_zone  temperature_c  vibration_mm_s  \
0      D001     storage         C           63.7             2.5   
1      D002      router         B           51.6             1.3   
2      D003      server         A           64.1             3.2   
3      D004     storage         C           65.6             4.0   
4      D005      router         B           63.1             3.8   

   operating_hours  error_count_7d  days_since_maintenance  \
0             9884             2.0                    83.0   
1             5822             4.0                   236.0   
2            12512             2.0                   219.0   
3             4703             3.0                   142.0   
4            12576             2.0                   157.0   

   downtime_hours_next_7d  failure_within_7d  
0                      23                  1  
1                      23                  1  
2                       4                  1  
3                       0     

### 8.3 学生任务

1. **准备特征**
   - 从模型特征中排除 `device_id`、`downtime_hours_next_7d` 和标签；
   - 将 `device_type`、`room_zone` 作为类别特征；
   - 其余输入字段作为数值特征。

In [20]:
# 1. 准备特征和标签
#类别特征
category_feature = ['device_type', 'room_zone']
#数值特征
numeric_features = [
    "temperature_c",
    "vibration_mm_s",
    "operating_hours",
    "error_count_7d",
    "days_since_maintenance",
]
#所有特征
feature_columns = category_feature + numeric_features
# device_id 只用于定位样本
# downtime_hours_next_7d 是未来信息，不能作为特征
X = devices[feature_columns]
y = devices['failure_within_7d']

2. **划分数据**
   - 按 `60% : 20% : 20%` 划分训练集、验证集和测试集；
   - 两次划分都使用 `random_state=42` 和 `stratify`；
   - 输出三个数据集的样本数和标签比例。

In [21]:
# 2. 按 60% : 20% : 20% 划分数据
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    # 40% 暂时保留给验证集和测试集
    test_size=0.4,
    # 固定随机种子，保证结果可以复现
    random_state=42,
    # 尽量保持原始标签比例
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp,
)

print("训练集、验证集、测试集样本数：")
print(len(X_train), len(X_valid), len(X_test))
print("训练集标签比例：")
print(y_train.value_counts(normalize=True).sort_index().round(3))
print("验证集标签比例：")
print(y_valid.value_counts(normalize=True).sort_index().round(3))
print("测试集标签比例：")
print(y_test.value_counts(normalize=True).sort_index().round(3))

训练集、验证集、测试集样本数：
54 18 18
训练集标签比例：
failure_within_7d
0    0.685
1    0.315
Name: proportion, dtype: float64
验证集标签比例：
failure_within_7d
0    0.667
1    0.333
Name: proportion, dtype: float64
测试集标签比例：
failure_within_7d
0    0.722
1    0.278
Name: proportion, dtype: float64


3. **建立 Pipeline**
   - 数值特征：中位数填充，再标准化；
   - 类别特征：众数填充，再进行 One-Hot 编码；
   - 分类器使用 `LogisticRegression(max_iter=1000, class_weight="balanced")`，让训练过程更加重视数量较少的故障类。

In [32]:
# 3. 建立预处理和分类 Pipeline
numeric_pipeline = Pipeline([
    (
        # "imputer" 是缺失值填充步骤的名称
        'imputer',
        # 使用训练集中每个数值字段的中位数填充缺失值
        SimpleImputer(strategy='median'),
    ),
    (
        # "scaler" 是标准化步骤的名称
        'scaler',
        # 标准化: 将不同量纲的数值特征调整到相近尺度
        StandardScaler(),
    )
])

category_pipeline = Pipeline([
    (
        'imputer',
        # 使用训练集中最常见的类别填充缺失值
        SimpleImputer(strategy='most_frequent'),
    ),
    (
        # "onehot" 是类别编码步骤的名称(0/1)
        'onehot',
        # 验证或测试时遇到新类别不报错
        OneHotEncoder(handle_unknown='ignore'),
    )
])

# 让不同字段进入各自的预处理流程
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('category', category_pipeline, numeric_features)
])

# 将全部预处理步骤和逻辑回归模型串联起来
device_model = Pipeline([
    # 先完成数值特征和类别特征的预处理
    ('preprocessor', preprocessor),
     # 再使用处理后的全部特征训练分类模型
    (
        'classifier',
        LogisticRegression(
            # 最多迭代 1000 次，避免模型未收敛就停止
            max_iter=1000,
            # 训练时更加重视数量较少的故障类
            class_weight='balanced'
        )
    )
])

# 只使用训练集学习预处理规则和模型参数
device_model.fit(X_train, y_train)

# 使用训练好的模型预测训练集
train_predictions = device_model.predict(X_train)
# 使用训练好的模型预测验证集
valid_predictions = device_model.predict(X_valid)

4. **评估验证集**
   - 比较训练集和验证集 Accuracy；
   - 判断两者是否存在明显差距；
   - 输出验证集混淆矩阵和 `classification_report()`；
   - 记录总体 Accuracy，以及正类的 Precision、Recall 和 F1。

In [31]:
# 4.比较训练集和验证集accuracy(准确率)
print(
    '训练集Accuracy: ',
    round(accuracy_score(y_train, train_predictions), 3),
)

print(
    '验证集Accuracy: ',
    round(accuracy_score(y_valid, valid_predictions), 3),
)

print('验证集混淆矩阵: ')
print(
    confusion_matrix(
        y_valid,
        valid_predictions,
        # 固定行列顺序为未故障 0、故障 1
        labels=[0, 1]
    )
)

print('验证集报告: ')
print(
    classification_report(
        # y_true：验证集中的真实标签
        y_valid,
        # y_pred：模型输出的预测标签
        valid_predictions,
        # 固定报告中的类别顺序
        labels=[0, 1],
        target_names=['未故障', '故障'],
        # 指标保留 3 位小数
        digits=3,
        # 指标分母为 0 时返回 0
        zero_division=0,
    )
)

训练集Accuracy:  1.0
验证集Accuracy:  0.722
验证集混淆矩阵: 
[[8 4]
 [1 5]]
验证集报告: 
              precision    recall  f1-score   support

         未故障      0.889     0.667     0.762        12
          故障      0.556     0.833     0.667         6

    accuracy                          0.722        18
   macro avg      0.722     0.750     0.714        18
weighted avg      0.778     0.722     0.730        18



In [33]:
# 验证集的 Precision、Recall 和 F1

#precision:精确率
valid_precision = precision_score(
    y_valid,
    valid_predictions,
    # 指定标签 1“故障”为正类
    pos_label=1,
    #分母为0时返回
    zero_division=0,
)

#recall:召回率
valid_recall = recall_score(
    y_valid,
    valid_predictions,
    pos_label=1,
    zero_division=0,
)

#f1: 平衡precision和recal
valid_f1 = f1_score(
    y_valid,
    valid_predictions,
    pos_label=1,
    zero_division=0,
)

5. **选择分类阈值**
   - 比较阈值 `0.30`、`0.50` 和 `0.70`；
   - 在满足 `Recall >= 0.60` 的候选值中选择 Precision 较高的一项；
   - 只根据验证集结果选择阈值。

In [34]:
# 6. 在验证集上比较分类阈值

# 找到标签 1“故障”对应的概率列
positive_class_index = list(device_model.classes_).index(1)
valid_probabilities = device_model.predict_proba(X_valid)[
    :, positive_class_index
]

# 比较三个候选阈值
for threshold in [0.30, 0.50, 0.70]:

    # 概率达到阈值时预测为 1，否则预测为 0
    threshold_predictions = (
        valid_probabilities >= threshold
    ).astype(int)

    #accuracy:精确度
    threshold_accuracy = accuracy_score(
        y_valid,
        threshold_predictions,
    )

    #precision:准确度
    threshold_precision = precision_score(
        y_valid,
        threshold_predictions,
        pos_label=1,
        zero_division=0,
    )

    #recall: 召回率
    threshold_recall = recall_score(
        y_valid,
        threshold_predictions,
        pos_label=1,
        zero_division=0,
    )

    #f1
    threshold_f1 = f1_score(
        y_valid,
        threshold_predictions,
        pos_label=1,
        zero_division=0,
    )

    print(
        f'阈值={threshold: 2f},'
        f'Accuracy: {threshold_accuracy:.2f},'
        f'Precision: {threshold_precision:.2f},'
        f'Recall: {threshold_recall:.2f},'
        f'F1: {threshold_f1:.2f}'
    )

阈值= 0.300000,Accuracy: 0.50,Precision: 0.40,Recall: 1.00,F1: 0.57
阈值= 0.500000,Accuracy: 0.72,Precision: 0.56,Recall: 0.83,F1: 0.67
阈值= 0.700000,Accuracy: 0.72,Precision: 0.67,Recall: 0.33,F1: 0.44


In [35]:
# 0.30 和 0.50 的 Recall 都达到 0.60
# 0.50 的 Precision 更高，因此选择 0.50
FINAL_THRESHOLD = 0.5
valid_predictions = (
    valid_predictions >= FINAL_THRESHOLD
).astype(int)

6. **分析错误样本**
   - 建立包含设备编号、真实标签、预测标签和故障概率的错误表；
   - 将错误标记为 `FN_漏报` 或 `FP_误报`；
   - 至少选择 2 条错误，完成“证据—原因假设—改进动作—验证方法”分析。

In [36]:
# 6. 建立验证集错误样本表
error_analysis = X_valid.copy()
error_analysis.insert(
    0,
    'device_id',
    devices.loc[X_valid.index, 'device_id'],
)

error_analysis['actual'] = y_valid
error_analysis["predicted"] = valid_predictions
error_analysis["failure_probability"] = valid_probabilities
error_analysis["is_error"] = (error_analysis["actual"] != error_analysis["predicted"])
errors = error_analysis[error_analysis["is_error"]].copy()
errors["error_type"] = "FP_误报"
errors.loc[errors["actual"] == 1, "error_type"] = "FN_漏报"

error_columns = [
    'device_id',
    'actual',
    'predicted',
    'failure_probability',
    'error_type'
]

print('验证集错误样本: ')
print(
    errors[error_columns]
    .sort_values('failure_probability')
    .round({'failure_probability': 3})
)

验证集错误样本: 
   device_id  actual  predicted  failure_probability error_type
1       D002       1          0                0.327      FN_漏报
30      D031       0          1                0.520      FP_误报
51      D052       0          1                0.564      FP_误报
56      D057       0          1                0.643      FP_误报
89      D090       0          1                0.728      FP_误报


7. **完成最终测试**
   - 模型和阈值确定后，只在测试集上评估一次；
   - 输出测试集混淆矩阵和分类报告；
   - 说明测试结果与验证结果是否一致，以及当前数据的限制。

In [37]:
# 7. 模型和阈值确定后，测试集只评估一次
test_probabilities = device_model.predict_proba(X_test)[
    :, positive_class_index
]
test_predictions = (
    test_probabilities >= FINAL_THRESHOLD
).astype(int)

print('测试集混淆矩阵: ')
print(
    confusion_matrix(
        y_test,
        test_predictions,
        labels=[0, 1]
    )
)

print('测试集分类报告: ')
print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=['未故障', '故障'],
        digits=3,
        zero_division=0
    )
)

测试集混淆矩阵: 
[[11  2]
 [ 3  2]]
测试集分类报告: 
              precision    recall  f1-score   support

         未故障      0.786     0.846     0.815        13
          故障      0.500     0.400     0.444         5

    accuracy                          0.722        18
   macro avg      0.643     0.623     0.630        18
weighted avg      0.706     0.722     0.712        18

